# «Trustworthy Online Controlled Experiments»

Ron Kohavi, Diane Tang, Ya Xu, 2020.

Ноутбук к разбору книги: https://dilyakhakimova.com/shelf/kohavi-trustworthy/

Симуляции ко всем визуализациям разбора, по главам: сколько идей срабатывает, выручка и интервалы в примере с промокодом, калькулятор MDE до и после эксперимента, закон Твимена, подглядывание, почему мерить при 50 на 50, сто метрик и три порога, тысяча A/A делением платформы и проверка SRM.

Перед каждой ячейкой написано, что она делает и что в ней поменять. Код и числа наши, по описанию методов из книги; нужны только Python и numpy. Ячейки независимые: запускай любую.

## Глава 1. Двадцать идей: сколько сработает

Что делает симуляция: 10 000 раз разыгрывает квартал из 20 идей. Каждая идея помогает с вероятностью из книги: треть в молодом продукте, 15% в зрелом. Печатает, сколько удачных идей обычно набирается и как редко удачна хотя бы половина. Поменяй p_win и ideas под свою команду.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)

# Доли из книги: в молодом продукте помогает примерно треть идей, в зрелом — 10–20%, здесь берём 15%
p_win = {"молодой продукт": 1 / 3, "зрелый продукт": 0.15}
ideas, runs = 20, 10_000          # 20 идей за квартал, 10 000 повторов квартала

for name, p in p_win.items():
    wins = rng.binomial(ideas, p, runs)          # сколько идей из 20 помогло в каждом повторе
    print(f"{name}: в среднем помогает {wins.mean():.1f} из {ideas}")
    for k in range(0, 13, 2):
        share = np.mean((wins >= k) & (wins < k + 2))
        print(f"  {k:>2}–{k + 1:<2} удачных  {'█' * round(share * 60)} {share:.0%}")
    print(f"  хотя бы половина удачных: {np.mean(wins >= ideas / 2):.1%}\n")

## Глава 2. Пустое поле промокода: выручка и интервалы

Что делает симуляция: создаёт три группы по 900 000 пользователей со средней выручкой из книги — 3,21 $, 3,12 $ и 2,96 $ — и считает для тестов разницу с контролем и её 95%-й доверительный интервал дельта-методом. Размер групп и устройство выручки наши, поэтому числа чуть отличаются от книги. Это та же схема, что на странице.

In [ ]:
import numpy as np

rng = np.random.default_rng(2)

# Средние из книги: выручка на пользователя, начавшего оформлять заказ — 3,21 $ в контроле, 3,12 $ и 2,96 $ в тестах.
# Размер групп и устройство выручки наши: 5% покупают, средний чек 64,2 $, разброс чека как у обычных покупок.
n = 900_000                                     # пользователей в каждой группе
check = 64.2 groups = {"Контроль": 3.21, "Тест 1": 3.12, "Тест 2": 2.96}

def revenue(mean):
    buy = rng.random(n) < mean / check          # купил или нет
    return buy * rng.gamma(2.0, check / 2.0, n)  # сумма покупки, в среднем check

data = {g: revenue(m) for g, m in groups.items()}
ctrl = data["Контроль"]
m_c, se_c = ctrl.mean(), ctrl.std(ddof=1) / np.sqrt(n)
print(f"{'группа':<9} {'выручка':>8} {'Δ%':>7}   95%-й интервал Δ%")
for g, x in data.items():
    m, se = x.mean(), x.std(ddof=1) / np.sqrt(n)
    if g == "Контроль":
        print(f"{g:<9} {m:>7.2f}$"); continue
    d = m / m_c - 1                                          # относительная разница
    se_d = (m / m_c) * np.sqrt((se / m) ** 2 + (se_c / m_c) ** 2)   # дельта-метод для отношения
    lo, hi = d - 1.96 * se_d, d + 1.96 * se_d
    print(f"{g:<9} {m:>7.2f}$ {d:>+7.1%}   [{lo:+.1%}, {hi:+.1%}]" + ("   ниже нуля целиком" if hi < 0 else ""))

## Глава 2. Калькулятор MDE до и после теста

До теста: какой минимальный эффект заметим за 7, 14 и 28 дней при таком трафике и сколько дней нужно под нужный эффект. После теста: какой эффект тест мог заметить на самом деле по фактическим размерам групп — если наблюдаемая разница меньше, «эффекта нет» значит «не умеем увидеть такой маленький». Подставь свою конверсию, трафик и итоги теста. То же самое без кода — калькулятор MDE на сайте: https://dilyakhakimova.com/play/sample-size/

In [ ]:
from statistics import NormalDist
from math import sqrt, ceil

z = NormalDist().inv_cdf

def mde_before(p, users_per_day, days, alpha=0.05, power=0.8, share_test=0.5):
    """До теста: какой минимальный эффект на конверсии p заметим за days дней при таком трафике."""
    n_t = users_per_day * days * share_test
    n_c = users_per_day * days * (1 - share_test)
    s2 = p * (1 - p)                                   # дисперсия доли
    return (z(1 - alpha / 2) + z(power)) * sqrt(s2 / n_t + s2 / n_c)

def days_needed(p, effect_rel, users_per_day, alpha=0.05, power=0.8):
    """До теста: сколько дней нужно, чтобы заметить относительный эффект effect_rel при сплите 50/50."""
    s2, d = p * (1 - p), p * effect_rel
    n_per_group = 2 * s2 * (z(1 - alpha / 2) + z(power)) ** 2 / d ** 2 return ceil(2 * n_per_group / users_per_day)

def mde_after(mean_a, sd_a, n_a, mean_b, sd_b, n_b, alpha=0.05, power=0.8):
    """После теста: какой эффект тест мог заметить на самом деле — по фактическим размерам групп и разбросу."""
    return (z(1 - alpha / 2) + z(power)) * sqrt(sd_a ** 2 / n_a + sd_b ** 2 / n_b)

# ---- до теста: подставь свою конверсию и трафик ----
p, traffic = 0.10, 20_000                              # конверсия 10%, 20 000 пользователей в день на тест
for d in [7, 14, 28]:
    m = mde_before(p, traffic, d)
    print(f"{d:>2} дней: заметим эффект от {100 * m:.2f} п. п., то есть от {m / p:.1%} относительно")
print(f"Чтобы заметить рост на 2%, нужно {days_needed(p, 0.02, traffic)} дней")

# ---- после теста: подставь итог своего теста (для доли sd = √(p·(1−p))) ----
pa, pb, na, nb = 0.100, 0.101, 140_000, 139_500 m = mde_after(pa, sqrt(pa * (1 - pa)), na, pb, sqrt(pb * (1 - pb)), nb)
print(f"\nПосле теста: при таких группах тест мог заметить эффект от {100 * m:.2f} п. п. ({m / pa:.1%}).")
print("Наблюдали", f"{100 * (pb - pa):+.2f} п. п. —", "меньше MDE: «эффекта нет» значит «не умеем увидеть такой маленький»"
      if abs(pb - pa) < m else "больше MDE")

## Глава 3. Закон Твимена: доля багов среди «побед»

Что делает симуляция: миллион экспериментов, у 3 из 100 баг. Честный результат почти всегда в пределах ±2%, баг сдвигает метрику куда угодно. Для каждого эффекта в отчёте печатает, какая доля экспериментов с таким эффектом — баги. Это та же модель, что в игрушке на странице; поменяй bug_share и разброс под свой продукт.

In [ ]:
import numpy as np

rng = np.random.default_rng(3)

# Игрушечная модель с нашими числами, как в игрушке на странице:
# у команды баг в 3 тестах из 100; честный результат ≈ N(+0,3%; 0,8%), результат с багом ≈ N(0%; 8%) — куда угодно.
tests, bug_share = 1_000_000, 0.03 bug = rng.random(tests) < bug_share
effect = np.where(bug, rng.normal(0, 8, tests), rng.normal(0.3, 0.8, tests))

print("эффект в отчёте   тестов с таким эффектом   из них с багом")
for lo in [0.5, 1, 2, 3, 5, 10]:
    near = np.abs(effect - lo) < 0.25                     # отчёты с эффектом около lo
    print(f"   около +{lo:<4}%   {near.sum():>10}              {bug[near].mean():>6.0%}")

## Глава 3. Подглядывание: 1000 A/A‑тестов

Что делает симуляция: 1000 A/A-тестов по 20 дней, эффекта нет ни в одном. Каждый день считает p-value по накопленным данным и сравнивает два правила: смотреть только в конце и остановиться в первый же день, когда стало «значимо». Печатает долю ложных срабатываний для обоих — около 5% против многократно большей. Поменяй days и посмотри, как растут ложные срабатывания.

In [ ]:
import numpy as np

rng = np.random.default_rng(1)
tests, days, per_day, conv = 1000, 20, 500, 0.10   # 1000 A/A-тестов по 20 дней, по 500 пользователей в день в группе

# накопленные заказы в группах A и B; эффекта нет: конверсия одна и та же
a = rng.binomial(per_day, conv, (tests, days)).cumsum(axis=1)
b = rng.binomial(per_day, conv, (tests, days)).cumsum(axis=1)
n = per_day * np.arange(1, days + 1)

pool = (a + b) / (2 * n)
se = np.sqrt(pool * (1 - pool) * 2 / n)
z = np.abs(b - a) / n / se
significant = z > 1.96          # «значимо» при α = 0.05 print(f"смотрим один раз, в конце:          {significant[:, -1].mean():.1%} ложных срабатываний")
print(f"смотрим каждый день, стоп при «значимо»: {significant.any(axis=1).mean():.1%} ложных срабатываний")

## Глава 15. Почему мерить при 50 на 50

Что делает симуляция: 100 000 пользователей делят на тест и контроль в разных пропорциях — от 50 на 50 до 1 на 99 — и 2000 раз считают разницу конверсий без всякого эффекта. Печатает, насколько шумит оценка: при 50 на 50 меньше всего, при 1% тест шумит в пять раз сильнее.

In [ ]:
import numpy as np

rng = np.random.default_rng(15)

# Почему мерить при 50 на 50. Пользователей всего n, доля теста q; эффекта нет, смотрим, как сильно шумит оценка разницы.
n, runs, p = 100_000, 2_000, 0.10              # 100 000 пользователей, 2000 повторов, конверсия 10%
print(" доля теста   разброс оценки Δ   во сколько раз хуже, чем 50 на 50")
base = None
for q in [0.5, 0.25, 0.10, 0.05, 0.01]:
    nt = int(n * q); nc = n - nt
    d = rng.binomial(nt, p, runs) / nt - rng.binomial(nc, p, runs) / nc
    sd = d.std()
    base = base or sd
    print(f"   {q:>5.0%}         {sd:.5f}            {sd / base:.1f}")
print("\nПо формуле разброс ∝ 1 / √(q·(1−q)): минимум ровно при q = 50%.")

## Главы 17–18. Сто метрик: один порог или три

Что делает симуляция: 10 000 раз собирает отчёт из 100 метрик без всякого эффекта и считает, сколько покраснеет случайно — с одним порогом 0,05 для всех и с тремя порогами по группам. Разбивка метрик 10, 30 и 60 — пример, поменяй её под свой отчёт.

In [ ]:
import numpy as np

rng = np.random.default_rng(17)

# Сотня метрик в отчёте, эффекта нет ни у одной — сколько покраснеет случайно?
# Разбивка 10 / 30 / 60 — пример: точно затронутые, возможно затронутые, вряд ли затронутые.
groups = {"точно затронутые": (10, 0.05), "возможно затронутые": (30, 0.01), "вряд ли затронутые": (60, 0.001)}
reports = 10_000 total_one, total_tiers = 0, 0 for name, (k, alpha) in groups.items():
    p = rng.random((reports, k))                 # p-value метрик без эффекта распределены равномерно
    one = (p < 0.05).sum(axis=1).mean()          # один порог 0,05 для всех
    tier = (p < alpha).sum(axis=1).mean()        # свой порог для группы
    total_one += one; total_tiers += tier
    print(f"{name:<20} {k:>3} метрик  порог {alpha:<5}  случайно красных: {tier:.2f}  (при 0,05 было бы {one:.2f})")
print(f"\nВсего случайно красных на отчёт: {total_tiers:.2f} с тремя порогами против {total_one:.2f} с одним")

## Глава 19. Тысяча A/A: деление как у платформы

Что делает код: берёт пользователей и метрику за прошлую неделю и тысячу раз делит их на тест и контроль той же функцией, что платформа, — хешем с новой солью для каждого A/A. Печатает, как колеблется доля теста, долю p < 0,05 — при честном делении около 5% — и гистограмму p-value, а тест Колмогорова — Смирнова проверяет, равномерна ли она. Функцию split замени на деление своей платформы, p_value — на её расчёт. Файла с данными пока нет, поэтому в примере выдуманные числа.

In [ ]:
import hashlib
import numpy as np
from math import erfc, sqrt, exp

# Данные: метрика за прошлую неделю, одна строка на пользователя: id и значение.
# Файла с данными пока нет, поэтому в примере выдуманные числа. Когда будет файл — раскомментируй две строки
# и удали две строки с примером:
# import pandas as pd
# df = pd.read_csv("metric_last_week.csv"); user_ids, values = df["user_id"].astype(str).to_numpy(), df["value"].to_numpy()
user_ids = np.array([f"u{i}" for i in range(10_000)])
values = np.random.default_rng(7).lognormal(mean=6, sigma=1.2, size=len(user_ids))

def split(user_id, salt):
    """Деление на группы. Замени на функцию своей платформы: какой хеш, как склеиваются соль и id, какой порог."""
    h = int(hashlib.md5(f"{salt}:{user_id}".encode()).hexdigest(), 16)
    return h % 100 < 50          # True — тест, False — контроль

def p_value(a, b):
    """Тот же расчёт, что в отчёте платформы. Здесь — разница средних, дисперсия по пользователям."""
    se = sqrt(a.var(ddof=1) / len(a) + b.var(ddof=1) / len(b))
    return erfc(abs(b.mean() - a.mean()) / se / sqrt(2))

# 1000 A/A: те же пользователи, но каждый раз новая соль, как у нового эксперимента
runs = 1000          # считается около минуты: хеш для каждого пользователя в каждом A/A
pv = np.empty(runs)
share = np.empty(runs)
for i in range(runs):
    in_test = np.array([split(u, f"aa-{i}") for u in user_ids])
    share[i] = in_test.mean()
    pv[i] = p_value(values[~in_test], values[in_test])

print(f"доля теста: от {share.min():.1%} до {share.max():.1%} — должна колебаться около 50%")
print(f"p < 0.05 у {np.mean(pv < 0.05):.1%} A/A — при честной платформе около 5%\n")

# гистограмма p-value: 20 столбиков по 0.05, у равномерного распределения каждый около 5%
counts, _ = np.histogram(pv, bins=20, range=(0, 1))
for k, c in enumerate(counts):
    print(f"{k * 0.05:.2f}–{(k + 1) * 0.05:.2f}  {'█' * round(c / runs * 400)} {c / runs:.1%}")

# тест Колмогорова — Смирнова: похоже ли распределение p-value на равномерное
x = np.sort(pv)
d = max(np.max(np.arange(1, runs + 1) / runs - x), np.max(x - np.arange(runs) / runs))
lam = (sqrt(runs) + 0.12 + 0.11 / sqrt(runs)) * d
p_ks = min(1.0, 2 * sum((-1) ** (k - 1) * exp(-2 * k * k * lam * lam) for k in range(1, 101)))
print(f"\nКолмогоров — Смирнов: D = {d:.3f}, p = {p_ks:.3f}",
      "— похоже на равномерное" if p_ks > 0.05 else "— не похоже на равномерное: ищи ошибку в делении или в расчёте")

## Глава 21. Проверка SRM χ²-критерием

Что делает код: сравнивает фактические размеры групп с задуманным сплитом критерием χ² и печатает p-value. Порог для SRM строгий — p < 0,0005, как советуют авторы: проверку запускают в каждом тесте, и при 0,05 тревога срабатывала бы зря в каждом двадцатом. Подставь в n_a и n_b числа из своего теста; если сплит не 50/50, задай долю контроля: например, при 90/10 — srm_check(n_a, n_b, share_a=0.9).

In [ ]:
import numpy as np
from math import erfc, sqrt

def srm_check(n_a, n_b, share_a=0.5):
    """χ²-критерий: совпадает ли соотношение групп с задуманным сплитом."""
    total = n_a + n_b
    exp_a, exp_b = total * share_a, total * (1 - share_a)
    chi2 = (n_a - exp_a) ** 2 / exp_a + (n_b - exp_b) ** 2 / exp_b
    p = erfc(sqrt(chi2 / 2))   # p-value для χ² с одной степенью свободы
    return chi2, p

# сплит 50/50, в группе B антифрод отрезал часть пользователей
n_a, n_b = 500_000, 493_000   # ← подставь свои числа
chi2, p = srm_check(n_a, n_b)
print(f"A = {n_a}, B = {n_b}, доля A = {n_a / (n_a + n_b):.4f}")
print(f"χ² = {chi2:.1f}, p-value = {p:.2g}")
print("SRM: отчёту не верим, ищем причину" if p < 0.0005 else "перекос в пределах шума")